In [11]:
import pandas as pd
import numpy as np
import random
import time

# Cargamos los datos

In [12]:
distance_matrix = np.genfromtxt('./data/gr17.csv', delimiter=",")
pheromone_matrix = np.ones_like(distance_matrix)

In [1]:
def select_next_city(current_city, unvisited_cities, pheromone_matrix, distance_matrix, alpha, beta): 
  prob_of_next = np.zeros_like(unvisited_cities, dtype=float)
  total_sum = 0

  # --- Calculamos el "atractivo" de cada ciudad posible ---
  for i in range(len(unvisited_cities)): 
    total_sum += (((pheromone_matrix[current_city][unvisited_cities[i]])**alpha) * 
                  ((1/distance_matrix[current_city][unvisited_cities[i]])**beta))

  
  
  # --- Calcular la probabilidad de cada ciudad ---
  for i in range(len(unvisited_cities)): 
    prob_of_next[i] = (((pheromone_matrix[current_city][unvisited_cities[i]])**alpha) * 
                  ((1/distance_matrix[current_city][unvisited_cities[i]])**beta)) / total_sum

  # --- Elegimos una ciudad utilizando el metodo de la ruleta ---
  random_number = random.random()
  acum_prob = 0

  for i, prob in enumerate(prob_of_next):
    acum_prob += prob
    if (random_number <= acum_prob): 
      return unvisited_cities[i]


In [2]:
def deposit_pheromones_Qcte(n_ants, n_cities, pheromone_matrix, tour_matrix, Qfer, _): 
  for i in range(n_ants):                               # Por hormiga
    delta_pheromones = np.zeros_like(pheromone_matrix)  # Matriz para guardar el aporte de esta hormiga

    # Recorremos cada paso del tour de la hormiga 'i'

    for j in range(-1, n_cities-1): 
      idx_act_city = int(tour_matrix[i][j])
      idx_next_city = int(tour_matrix[i][j+1])

       # Sumamos una cantidad constante 'Qfer' a la arista recorrida.
      delta_pheromones[idx_act_city][idx_next_city] += Qfer
      # Lo hacemos en ambos sentidos porque el camino es simétrico (distancia A->B = B->A)
      delta_pheromones[idx_next_city][idx_act_city] += Qfer

    pheromone_matrix += delta_pheromones
  return pheromone_matrix

In [15]:
def deposit_pheromones_Qvar(n_ants, n_cities, pheromone_matrix, tour_matrix, Qfer, distance_matrix): 
  for i in range(n_ants): 
    delta_pheromones = np.zeros_like(pheromone_matrix) 

    for j in range(-1, n_cities-1): 
      idx_act_city  = int(tour_matrix[i][j])
      idx_next_city = int(tour_matrix[i][j+1])

      dist = distance_matrix[idx_act_city][idx_next_city]

      # La cantidad de feromona es inversamente proporcional a la distancia.
      # A menor 'dist', mayor la cantidad depositada.
      delta_pheromones[idx_act_city][idx_next_city] += Qfer / dist
      delta_pheromones[idx_next_city][idx_act_city] += Qfer / dist # Simétrico
    pheromone_matrix += delta_pheromones

  return pheromone_matrix

In [6]:
def deposit_pheromones_Qltot(n_ants, n_cities, pheromone_matrix, tour_matrix, Qfer, distance_matrix):
    for i in range(n_ants):                                     # Por cada hormiga
        
        # --- Calcular la longitud total del recorrido de esta hormiga ---
        total_dist = 0
        for j in range(-1, n_cities-1):
            idx_act_city  = int(tour_matrix[i][j])
            idx_next_city = int(tour_matrix[i][j+1])
            total_dist += distance_matrix[idx_act_city][idx_next_city]
            
        # --- Depositar feromona en base a esa longitud total ---
        delta_pheromones = np.zeros_like(pheromone_matrix)
        for j in range(-1, n_cities-1):
            idx_act_city  = int(tour_matrix[i][j])
            idx_next_city = int(tour_matrix[i][j+1])
            
            # Todas las aristas de este tour se refuerzan con un valor que depende de 'total_dist'.
            # Si la hormiga encontró un tour corto (bajo 'total_dist'), el refuerzo será grande.
            delta_pheromones[idx_act_city][idx_next_city] += Qfer / total_dist
            delta_pheromones[idx_next_city][idx_act_city] += Qfer / total_dist # Simétrico
            
        # Sumamos el aporte a la matriz global
        pheromone_matrix += delta_pheromones
        
    return pheromone_matrix

In [8]:
def min_distance(tour_matrix, distance_matrix): 
  # Inicializamos la distancia mínima con un valor infinito.
  # Cualquier distancia real será menor que este valor.
  min_dist = float('inf')
  best_tour_for_iteration = None # Para guardar el mejor recorrido de esta vuelta
  # Iteramos sobre cada hormiga (cada fila de la tour_matrix)
  for k in range(tour_matrix.shape[0]):
    # 'tour_k' es el recorrido de la hormiga 'k', ej: [0, 15, 8, ...]
    tour_k = tour_matrix[k]
    dist_k = 0 # Inicializamos la distancia para esta hormiga

    # Calculamos la distancia total de su recorrido
    # Sumamos las distancias de un paso al siguiente
    for i in range(len(tour_k) - 1):
      dist_k += distance_matrix[tour_k[i]][tour_k[i+1]]

    # Sumamos la distancia de vuelta a la ciudad de inicio para cerrar el ciclo.
    dist_k += distance_matrix[tour_k[-1]][tour_k[0]]

    # Comparamos si esta hormiga fue mejor que la mejor encontrada hasta ahora
    if(dist_k < min_dist):
      min_dist = dist_k  # Actualizamos la distancia mínima
      best_tour_for_iteration = tour_k # Guardamos su recorrido

  return best_tour_for_iteration, min_dist


In [29]:
# --- 1. Inicialización de Parámetros Generales ---
n_ants = 15                            # Número de hormigas
max_it = 100                           # Número de iteraciones (generaciones)
evap_var = 0.3                         # Tasa de evaporación de feromonas (ρ)
Qfer = 1                               # Cantidad base de feromonas a depositar
alpha = 1                              # Importancia de la feromona
beta = 1                               # Importancia de la distancia (visibilidad)
n_cities = distance_matrix.shape[0]    # Número de ciudades (obtenido de la matriz)

# Lista de las funciones de depósito que queremos probar
deposit_func = [deposit_pheromones_Qcte, deposit_pheromones_Qvar, deposit_pheromones_Qltot]

print("Iniciando simulación del Sistema de Hormigas para el Problema del Viajante...")
print("-----------------------------------------------------------------------------")

# --- 2. Bucle Principal del Experimento ---
# Este bucle ejecutará el algoritmo completo 3 veces, una para cada función de depósito.
for k in range(len(deposit_func)):
    
    # --- Reinicio para cada experimento ---
    pheromone_matrix = np.ones_like(distance_matrix) # Reiniciamos las feromonas
    best_overall_tour = [None, float('inf')]       # [recorrido, distancia] para el mejor de la historia
    
    inicio = time.time() # Empezamos a medir el tiempo

    # --- Bucle de Iteraciones (el corazón del algoritmo) ---
    for it in range(max_it):
        # Matriz para guardar los recorridos de las 30 hormigas en ESTA iteración
        tour_matrix = np.zeros((n_ants, n_cities), dtype=int)

        # --- Bucle de Hormigas ---
        for i in range(n_ants):
            current_city = 0  # Todas las hormigas empiezan en la ciudad 0
            unvisited_cities = list(range(1, n_cities)) # Ciudades pendientes de visitar
            tour_matrix[i, 0] = current_city # El primer paso del tour es la ciudad 0
            
            # Construimos el resto del recorrido para la hormiga 'i'
            for j in range(1, n_cities):
                next_city = select_next_city(current_city, unvisited_cities, pheromone_matrix,
                                             distance_matrix, alpha, beta)
                
                tour_matrix[i, j] = int(next_city)
                unvisited_cities.remove(next_city)
                current_city = next_city
        
        # --- Actualización de Feromonas (Evaporación + Depósito) ---
        pheromone_matrix *= (1 - evap_var)
        pheromone_matrix = deposit_func[k](n_ants, n_cities, pheromone_matrix, tour_matrix, Qfer, distance_matrix)
        
        # --- Evaluación y Actualización del Mejor Recorrido ---
        # Buscamos al mejor de la iteración actual
        vec, distance = min_distance(tour_matrix, distance_matrix)
        
        # Si el mejor de esta iteración es mejor que el mejor histórico, lo guardamos
        if(best_overall_tour[1] > distance):
            best_overall_tour[1] = distance
            best_overall_tour[0] = vec

    fin = time.time() # Dejamos de medir el tiempo

    # --- 3. Impresión de Resultados ---
    nombre_funcion = deposit_func[k].__name__
    print(f"Método: {nombre_funcion}")
    print(f"  - Mejor recorrido: {best_overall_tour[0]}")
    print(f"  - Distancia mínima: {best_overall_tour[1]}")
    print(f"  - Tiempo de ejecución: {fin - inicio:.4f} segundos.\n")

print("-----------------------------------------------------------------------------")
print("Simulación finalizada.")

Iniciando simulación del Sistema de Hormigas para el Problema del Viajante...
-----------------------------------------------------------------------------
Método: deposit_pheromones_Qcte
  - Mejor recorrido: [ 0 12  3  6  7  5 16 13 14  2 10  9  1  4  8 11 15]
  - Distancia mínima: 2094.0
  - Tiempo de ejecución: 0.2692 segundos.

Método: deposit_pheromones_Qvar
  - Mejor recorrido: [ 0  3 12  6  7  5 16 13 14  2 10  4  1  9  8 11 15]
  - Distancia mínima: 2149.0
  - Tiempo de ejecución: 0.2543 segundos.

Método: deposit_pheromones_Qltot
  - Mejor recorrido: [ 0 15 11  8  4  1  9 10  2 14 13 16  5  7  6 12  3]
  - Distancia mínima: 2085.0
  - Tiempo de ejecución: 0.2690 segundos.

-----------------------------------------------------------------------------
Simulación finalizada.
